# Convolutional Neural Networks, Part 1: How Convolution Works

CSCI 6379 · Topic 20.

A dense layer was the wrong tool for images: it is blind to the facts that patterns are small, that the same pattern recurs elsewhere, and that fine detail is redundant. This notebook builds the fix from scratch. We price a dense layer against a convolutional one, hand-code a 2D convolution with two filters over a small binary image, nail the size arithmetic that breaks real implementations, add pooling, and finish with a small PyTorch CNN. Everything runs on Colab CPU or GPU.

## A dense layer is the wrong tool: the parameter cost

A small colour photograph of `32 x 32` pixels is `32 x 32 x 3 = 3,072` numbers. Wire that to a modest hidden layer of 512 units and the first layer alone costs `3,072 x 512 + 512 = 1,573,376` weights, for an image smaller than a postage stamp.

A convolutional layer taking a 3-channel image and using 32 filters of size `3 x 3` holds `32 x (3 x 3 x 3) + 32 = 896` weights instead. A filter is always as deep as its input, so each filter is `3 x 3 x 3`, and the number of filters becomes the output depth. Let the arithmetic speak.

In [ ]:
# Dense first layer on a 32x32x3 image, into 512 hidden units.
inputs = 32 * 32 * 3               # 3,072 numbers per image
dense_weights = inputs * 512 + 512 # weights + biases
print("dense inputs        :", inputs)
print("dense 3072 -> 512    :", f"{dense_weights:,}", "weights")

# A conv layer: 32 filters, each 3x3 and as deep as the 3-channel input.
conv_weights = 32 * (3 * 3 * 3) + 32   # weights per filter x filters, + biases
print("conv 32 filters 3x3x3:", f"{conv_weights:,}", "weights")

print("ratio               :", round(dense_weights / conv_weights), "times fewer")

## Hand-coding a 2D convolution

A filter (or kernel) is a small grid of weights. You lay it over a patch of the image, multiply each weight by the pixel under it, and sum: that gives one number. Slide one step and repeat. Over all valid positions a `k x k` filter on a `6 x 6` image produces a `4 x 4` feature map, a map of *where in the image this particular pattern occurs*.

We use a `6 x 6` binary image and two hand-written `3 x 3` filters:

- **F1** has `1`s on the diagonal and `-1`s elsewhere: a **diagonal-line detector**.
- **F2** has `1`s down its middle column: a **vertical-line detector**.

At the top-left corner the patch is a perfect diagonal, so every `1` in F1 meets a `1` in the image and every `-1` meets a `0`, giving `3`, the maximum this filter can produce. In a real CNN nobody chooses these nine numbers; they are learned by backpropagation. We wrote them by hand only to make the example readable.

In [ ]:
import numpy as np

IMG = np.array([
    [1, 0, 0, 0, 0, 1],
    [0, 1, 0, 0, 1, 0],
    [0, 0, 1, 1, 0, 0],
    [1, 0, 0, 0, 1, 0],
    [0, 1, 0, 0, 1, 0],
    [0, 0, 1, 0, 1, 0]], float)

F1 = np.array([[ 1, -1, -1],
               [-1,  1, -1],
               [-1, -1,  1]], float)   # diagonal detector
F2 = np.array([[-1,  1, -1],
               [-1,  1, -1],
               [-1,  1, -1]], float)   # vertical-line detector

def conv2d(img, f, stride=1):
    """Valid (no-padding) 2D cross-correlation, the operation CNNs call convolution."""
    k = f.shape[0]
    n = (img.shape[0] - k) // stride + 1
    return np.array([[float((img[i*stride:i*stride+k, j*stride:j*stride+k] * f).sum())
                      for j in range(n)] for i in range(n)])

FM1 = conv2d(IMG, F1)
FM2 = conv2d(IMG, F2)

print("feature map, filter 1 (diagonal detector):\n", FM1)
print("max of filter 1 map:", FM1.max(),
      "at", np.unravel_index(FM1.argmax(), FM1.shape))
print("\nfeature map, filter 2 (vertical detector):\n", FM2)

The `3` at the top-left of the first map is the perfect diagonal match; slide one step right and the match is gone. F2 lights up in completely different places. Stack the two `4 x 4` maps and you have a `2 x 4 x 4` output: two channels, one per filter. The filter weights are identical at every position, which is **weight sharing** and the whole point.

## Size arithmetic: getting it right in code

This is where implementations actually break. Sliding a `k x k` filter over a `W x W` input without padding gives `W - k + 1`; with `k = 3` that is `W - 2`, so **every convolution shrinks the image by two pixels**. **Padding** a ring of zeros fixes both the shrink and the under-use of border pixels. The general formula with padding `p` and stride `s` is

$$W_{\text{out}} = \left\lfloor \frac{W + 2p - k}{s} \right\rfloor + 1$$

Three cases you will use constantly, all with `k = 3` on a `W = 5` input, reproduce the note's `5 -> 3`, `5 -> 5`, `5 -> 2`.

In [ ]:
import math

def w_out(W, k, p, s):
    return math.floor((W + 2*p - k) / s) + 1

k, W = 3, 5
print("formula: W_out = floor((W + 2p - k) / s) + 1\n")
print(f"no padding   p=0 s=1 : {W} -> {w_out(W, k, 0, 1)}   (shrinks by 2 every layer)")
print(f"padding = 1  p=1 s=1 : {W} -> {w_out(W, k, 1, 1)}   (size preserved, the usual default)")
print(f"stride  = 2  p=0 s=2 : {W} -> {w_out(W, k, 0, 2)}   (roughly halves it)")
print("\npadding = (k-1)//2 preserves size for any odd k:", (k - 1) // 2)

The middle case is why almost every `3 x 3` convolution in real code uses `padding=1`: it makes the convolution shape-neutral, so only pooling changes the size. Confirm it in PyTorch, and note the tensor layout: PyTorch expects **(batch, channels, height, width)**, channels *before* the spatial dimensions. Getting that backwards is the single most common beginner error.

In [ ]:
import torch
import torch.nn as nn

conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
x = torch.randn(1, 3, 32, 32)   # batch, channels, height, width
print("input :", tuple(x.shape))
print("output:", tuple(conv(x).shape), " <- size preserved by padding=1")

## Pooling

Fine detail is largely redundant. A `2 x 2` **max-pool** slides a `2 x 2` window with stride 2 and keeps only the largest value in each block. It halves height and width (quartering the count), has **no parameters at all**, and makes the representation slightly invariant to small shifts: a feature map says "this pattern occurs here", and after pooling it says "this pattern occurs somewhere around here". Max is the common choice because the strongest detection in a neighbourhood is the informative one.

In [ ]:
def maxpool(M, k=2):
    n = M.shape[0] // k
    return np.array([[M[i*k:(i+1)*k, j*k:(j+1)*k].max() for j in range(n)]
                     for i in range(n)])

pooled = maxpool(FM1, 2)
print("feature map 4 x 4:\n", FM1)
print("\nafter 2 x 2 max pooling -> 2 x 2:\n", pooled)

## The whole thing: a small PyTorch CNN

Put the pieces in sequence: alternating convolution and pooling to build up features, then flatten and hand the result to ordinary dense layers. Because every convolution uses `padding=1`, the convolutions never change the size and **only the pools do**, which keeps the flatten size predictable: `32 -> 32 -> 16 -> 16 -> 8`, so with 64 channels the flattened vector is `64 x 8 x 8 = 4096`.

The place shapes actually bite you is the jump to the first `nn.Linear`, which needs an exact input size. Rather than compute `4096` by hand (it breaks silently when you add a layer), ask the network with a dummy forward pass, or let `nn.LazyLinear` infer it on first use.

In [ ]:
# Build the convolutional feature extractor as an nn.Sequential.
features = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),   # 32x32 -> 32x32
    nn.MaxPool2d(2),                             #       -> 16x16
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),  # 16x16 -> 16x16
    nn.MaxPool2d(2),                             #       -> 8x8
)

# Habit 1: ask the network for the flatten size instead of computing it in your head.
with torch.no_grad():
    n_flat = features(torch.zeros(1, 3, 32, 32)).flatten(1).shape[1]
print("flatten size from a dummy forward pass:", n_flat, "(= 64 x 8 x 8)")

classifier = nn.Sequential(
    nn.Flatten(),
    nn.Linear(n_flat, 512), nn.ReLU(),
    nn.Linear(512, 10),
)
model = nn.Sequential(features, classifier)

# Sanity check on a dummy CIFAR-sized batch: 8 images, 10 class scores each.
out = model(torch.randn(8, 3, 32, 32))
print("model output shape:", tuple(out.shape))

Habit 2 is `nn.LazyLinear`, which infers `in_features` from the first batch, so you never write the number at all.

In [ ]:
# Same head, but the first dense layer infers its input size on the first forward pass.
lazy_classifier = nn.Sequential(
    nn.Flatten(),
    nn.LazyLinear(512), nn.ReLU(),   # in_features inferred from the first batch
    nn.Linear(512, 10),
)
lazy_model = nn.Sequential(features, lazy_classifier)
print("before first pass:", lazy_classifier[1])   # in_features is still uninitialised
_ = lazy_model(torch.randn(1, 3, 32, 32))          # triggers inference
print("after  first pass:", lazy_classifier[1])    # now in_features = 4096

Finally, count where the weights live. All the seeing is done by less than one percent of them: the convolutions extract every feature and cost about 19 thousand weights, while the dense classifier that reads them off costs two million.

In [ ]:
def count(module):
    return sum(p.numel() for p in module.parameters())

conv_params  = count(features)
dense_params = count(classifier)
total        = conv_params + dense_params

print(f"both convolutional layers : {conv_params:>10,}   {conv_params/total:6.1%}")
print(f"both dense layers         : {dense_params:>10,}   {dense_params/total:6.1%}")
print(f"total                     : {total:>10,}")